In [ ]:
import json
import pandas as pd
import glob

In [ ]:
data_path_dict = {}
for path in glob.glob("data/19.10.2021/*.csv"):
    client = path.split("/")[-1].replace(".csv", "")
    data_path_dict[client] = path
print(data_path_dict)

In [ ]:
dfs_dict = {}
for client in data_path_dict:
    data = pd.read_csv(data_path_dict[client])
    for col in ["timestamp", "anomalyStart", "anomalyEnd"]:
        data[col + "_datetime"] = pd.to_datetime(data[col], unit="ms")
    dfs_dict[client] = data

In [ ]:
def total_regions(data):
    min_dt = min(data["anomalyStart_datetime"])
    max_dt = max(data["anomalyEnd_datetime"])
    diff = max_dt - min_dt
    print("Total number of regions", len(data), "over the period:", diff)


def avg_regions_per_day(data):
    avg_regions_per_day = data.set_index("anomalyStart_datetime").resample("D").count()
    print(
        "Average number of regions per day:",
        round(avg_regions_per_day["ITComponentId"].mean()),
    )


#     avg_regions_per_day["ITComponentId"].plot(kind="bar", figsize=(12, 4))


def avg_regions_per_day_unique_start_time(data):
    avg_regions_per_day = (
        data.drop_duplicates(subset="anomalyStart")
        .set_index("anomalyStart_datetime")
        .resample("D")
        .count()
    )
    print(
        "Average number of regions per day with unique start time:",
        round(avg_regions_per_day["ITComponentId"].mean()),
    )


#     avg_regions_per_day["ITComponentId"].plot(kind="bar", figsize=(12, 4))


def number_merged_intervals(df):
    intervals = [
        [(doc["anomalyStart"], "start"), (doc["anomalyEnd"], "end")]
        for doc in df.to_dict(orient="records")
    ]

    num_intervals = 0
    current_status = 0
    for t in sorted([i for sublist in intervals for i in sublist], key=lambda x: x[0]):
        if t[1] == "start":
            current_status += 1
        else:
            current_status -= 1
        if current_status == 0:
            num_intervals += 1
    #     print("Number of merged intervals:", num_intervals)
    return num_intervals


def unique_merged_regions_per_day(data):
    df = data.copy()
    df["date"] = df["anomalyStart_datetime"].apply(lambda x: x.date())
    n_days = 0
    regions = 0
    for group, temp_df in df.groupby(["date"]):
        regions += number_merged_intervals(temp_df)
        n_days += 1
    #         print(group, number_merged_intervals(temp_df))
    print("Average number of merged regions per day:", round(regions / n_days))


def number_regions_longer_than(data, duration):
    df = data.copy()
    df["duration"] = df["anomalyEnd"] - df["anomalyStart"]
    print(
        f"Number of regions longer than {duration} second(s) is {len(df[df['duration'] > duration])}"
    )

In [ ]:
unique_merged_regions_per_day(dfs_dict['cw'])

## Plotting SIRE regions

In [ ]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

In [ ]:
dfs_dict['cw'].columns

In [ ]:
def plot_regions(data):
    total_regions(data)
    avg_regions_per_day(data)
    avg_regions_per_day_unique_start_time(data)
    unique_merged_regions_per_day(data)
    yellow_color = "rgba(250, 250, 0, 0.75)"
    orange_color = "rgba(250, 160, 0, 0.75)"
    red_color = "rgba(250, 0, 0, 0.75)"
    black_color = "rgba(0, 0, 0, 0.5)"
    max_val = 1

    plot_number = data.ITComponentId.unique().size
    #     print("plot_number", plot_number)
    components = tuple(sorted(data.ITComponentId.unique()))
    title_str = tuple(
        f"{a}: {b} region(s)"
        for a, b in zip(
            components,
            map(
                number_merged_intervals,
                [data[data["ITComponentId"] == comp_name] for comp_name in components],
            ),
        )
    )
    #     print(title_str)

    fig = make_subplots(
        rows=plot_number,
        cols=1,
        shared_xaxes=True,
        #         vertical_spacing=0.02,
        specs=[[{"type": "scatter"}],] * plot_number,
        row_heights=[1.0] * plot_number,
        subplot_titles=title_str,
    )

    chart_idx = 1
    #     for comp_name, df in data.groupby("ITComponentId"):
    for comp_name in components:
        df = data[data["ITComponentId"] == comp_name]
        incident_cnt = 0
        traces = []
        for _, row in df.iterrows():
            incident_start_dt = row["anomalyStart_datetime"]
            incident_end_dt = row["anomalyEnd_datetime"]
            incident_score = row["relevanceScore"]
            metric_name = row["metricType"]
            if incident_score < 0.8:
                fill_color = yellow_color
            elif incident_score < 0.9:
                fill_color = orange_color
            else:
                fill_color = red_color

            display_txt = f"start: {incident_start_dt} end: {incident_end_dt} score: {round(incident_score, 2)} metric: {metric_name}"
            cur_trace = go.Scatter(
                x=[
                    incident_start_dt,
                    incident_start_dt,
                    incident_end_dt,
                    incident_end_dt,
                ],
                y=[0, max_val, max_val, 0],
                mode="lines",
                name=f"incident {incident_cnt}",
                text=display_txt,
                fill="tozeroy",
                fillcolor=fill_color,
                line_color=fill_color,
            )
            traces.append(cur_trace)
            incident_cnt += 1

        for trace in traces:
            fig.add_trace(trace, row=chart_idx, col=1)
        chart_idx += 1

    fig.update_layout(
        autosize=True,
        height=100 * plot_number,
        title="SIRE regions",
        clickmode="event+select",
        showlegend=False,
    )
    fig.update_yaxes(automargin=True)
    fig.show()

In [ ]:
client = "affluences"

plot_regions(dfs_dict[client][dfs_dict[client]["relevanceScore"] > 0.7])

In [ ]:
# data["ITComponentId"].unique().size

In [ ]:
# plot_regions(
#     data[
#         data["ITComponentId"].apply(
#             lambda x: x
#             in [
#                 "nodeheadless-browser-service-master",
#                 "dead-letter-celery-develop",
#                 "packetaipacket-ai-agent-service-master",
#             ]
#         )
#     ]
# )